# Notebook for plotting Rufu & Canup (2020) evolution Movies S4-S6 to accompany Figures 9 and S2-S4
## Preamble
### Load packages required

In [1]:
#SJL 1/2020
#Script to plot the change in surface length for planets for an example tidal evolution
#plots on a 2D map showing the change in shape and surface



###########################################################
###########################################################
###########################################################
import numpy as np
import scipy as sp
import sys
import os
import struct
from scipy import constants as const

from scipy.signal import savgol_filter

#package to use wildcards 
import fnmatch

import csv

from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import griddata

from scipy import interpolate

#plotting packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import pylab
import matplotlib.cm as cm
from matplotlib import gridspec


cwd = os.getcwd()
print(cwd)
if sys.platform== 'darwin':
    sys.path.insert(0, cwd+'/Support_scipts')
    print(cwd+'/Support_scripts')
elif (sys.platform== 'win32') | (sys.platform== 'win64'):
    sys.path.insert(0,cwd+"\\Support_scipts")
    
#HERCULES_structures
from HERCULES_structures import *
from surface_size_calc import *
from HERCULES_random_planet_database_structure_1D import *

#functions for calculating non-evenly spaced numerical differentials
from gradients import *

#import colormaps
import colormaps as cmaps
import matplotlib.cm as cm

import svglib.svglib as svglib
svglib.register_font('helvetica', './Helvetica.ttc')

/Users/vq21447/Documents/Lock_2026_SI
/Users/vq21447/Documents/Lock_2026_SI/Support_scripts
CHECK THE LOCATION OF ODYSSEY BACKUP
CHECK THE LOCATION OF ODYSSEY BACKUP


('helvetica', True)

### Define constants

In [ ]:
########################################################################################
########################################################################################
########################################################################################
#CONSTANTS
MEarth=5.972E24
LEM=3.5E34
REarth=6.371E6
MMoon=7.34767309E22

aMoon=0.3844E9
aCassini=30*REarth
aRoche=2.9*REarth

#for HERCULES
MEarth_H=5.9879648E24
LEM_H=3.53E34

### Set parameters and scenario to plot

In [ ]:
########################################################################################
########################################################################################
########################################################################################
#PARAMS
#info for HERCULES arrays
Hdir='Earth_correct_params_S3.20c'
Hname='Earth_correct_params_S3.20c'
    
#which plot do you want
#0: A=5, tau=49E5, Q/k=406 (Movie S4)
#1: A=500, tau=200E5, Q/k=406 (Movie S5)
#2: A=10000, tau=1500E5, Q/k=406 (Movie S6)
flag_data=2

#number of contor points
Ncont=1000 #100

#time step for plotting
Nstep_plot=500 

#Earth's moment of inertia used to convert to AM
C_Earth=0.335


#Give data files
dir='Data/Rufu&Canup_2020_quasi_resonance/ResultsEvection_2'
file_names=['EvolutionA5_Tau49.0e5_Phi0.txt',
            'EvolutionA500_Tau200.0e5_Phi0.txt',
            'EvolutionA10000_Tau1500.0e5_Phi0.txt']


#directory to find procesed length data
data_dir='Data/Rufu&Canup_2020_quasi_resonance'

#output file root
data_output_file_root='Rufu&Canup_surf_change'

#output for snapshots
if flag_data==0:
    output_dir='Movie_slides/MovieS4_slides'
elif flag_data==1:
    output_dir='Movie_slides/MovieS5_slides'
elif flag_data==2:
    output_dir='Movie_slides/MovieS6_slides'

if os.path.isdir(output_dir)==False:
    os.mkdir(output_dir)

## Main script
### Read in HERCULES database

In [ ]:
########################################################################################
########################################################################################
########################################################################################
#MAIN

########################################################################################
#read in the database

Hdatabase=HERCULES_random_planet_database_1D()
Hdatabase.make_array(Hdir,Hname)
Hdatabase.initialize_interpolation([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1],\
                                   [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]], flag_extrap=1)

#extract the latitudes for each Mu point
Nmu=Hdatabase.parr[0].Nmu
lat=np.arccos(Hdatabase.parr[0].layers[0].mu)*180/np.pi

### Read in orbital data

In [5]:

fdata=open(dir+'/'+file_names[flag_data])

reader = csv.reader(fdata, delimiter="\t", skipinitialspace=True)
temp = list(reader)

Ctime=[]
CkQ=[]
Ca=[]
Comg=[]
Comg_moon=[]
Ce=[]
Cphi=[]


for i in np.arange(len(temp)):
    
    if i==0:
        Aset=temp[i][0]
        Aset=float(Aset[6:])

    elif i==1:
        tau_set=temp[i][0]
        tau_set=float(tau_set[13:])

    elif (i>3)&(np.size(temp[i])>3):

        Ctime.append(temp[i][0]) #time
        Comg.append(temp[i][1]) #rotation rate of Earth
        Comg_moon.append(temp[i][2]) #rotation rate of Moon
        Ce.append(temp[i][3]) #eccentricity
        Ca.append(temp[i][4]) #semi-major axis
        Cphi.append(temp[i][5]) #Phi parameter related to tidal lag
        
        
#convert to useful type and units
omg_norm=np.sqrt(const.G*MEarth/(REarth**3))
Ctime=np.asarray(Ctime, dtype=np.float64)*1E4
Ca=np.asarray(Ca, dtype=np.float64)*REarth
Ce=np.asarray(Ce, dtype=np.float64)
Comg=np.asarray(Comg, dtype=np.float64)*omg_norm
Comg_moon=np.asarray(Comg, dtype=np.float64)*omg_norm

#calculate the AM and de-normalize
CL_tot=Comg/omg_norm+1.07E-3*Comg_moon/omg_norm+0.0367*np.sqrt(Ca/REarth*(1-Ce**2))
CL_tot=CL_tot*C_Earth*MEarth*REarth**2*omg_norm

CL=Comg*C_Earth*MEarth*REarth**2
     

print('done')

done


### Read in the precomputed strain data

In [6]:
#if needed read in the max integrated deformation
print('start')

#define the output files
data_output_file_int=data_dir+'/'+data_output_file_root+'_max_integrated_A_'+str(Aset)+'_tau_'+str(tau_set/1E5)+'E5.bin'

dataf_int = open(data_output_file_int, "rb")

#now read in the rest of the file as one massive array
ndata_per_step=6
data = np.fromfile(dataf_int, dtype=np.float64, count=-1)
dataf_int.close()

#work out how many time steps we have
temp=np.size(data)*1.0/(1.0*ndata_per_step)
if abs(temp-int(temp))<(1E-12):
    Ntstep=int(temp)
else:
    print("ERROR IN READ",'\n',"Not complete number of steps or incorrect number of print params",'\n',"EXITING")

#reshape array so that each row is a timestep
data=data.reshape(Ntstep,ndata_per_step)

steps=data[:,0].astype(int)
time=data[:,1]
ddl_lon_dt_max=data[:,2]
ddl_lon_dt_min=data[:,3]
ddl_lat_dt_min=data[:,4]
ddl_lat_dt_max=data[:,5]

print('end')
    
    

start
end


### Interpolate the orbital data to create the time snapshots

In [7]:
#Interpolate to find the time, AM and a to plot

#find the tstep based on length of evolution
tstep_plot=Ctime[-1]/Nstep_plot/1E6

time=np.arange(0.0,np.amax(Ctime),tstep_plot*1E6)
temp=np.where(time<np.amin(Ctime))[0]
if np.size(temp)!=0:
    time=time[(temp[-1]+1):]

Nt_plot=np.size(time) #number of time points to plot

#create 1D interpolation hulls for L and a
fCL = interpolate.interp1d(Ctime, CL, kind='linear')
fCa = interpolate.interp1d(Ctime, Ca, kind='linear')

#interpolate to find the L and a to plot
CL_plot=fCL(time)
Ca_plot=fCa(time)

print('done')
             

done


### Calculate the corresponding change in length

In [8]:
#now run through all time steps and calculate the change in length
print('begin')

checkpoints=np.linspace(1,101,101)

#run through and extract lengths and areas at each time point
dl_lat=np.zeros((Nt_plot,Nmu))
dl_lon=np.zeros((Nt_plot,Nmu))
dA=np.zeros((Nt_plot,Nmu))

ddl_lat_dL=np.zeros((Nt_plot,Nmu))
ddl_lon_dL=np.zeros((Nt_plot,Nmu))
ddA_dL=np.zeros((Nt_plot,Nmu))

ddl_lat_dt=np.zeros((Nt_plot,Nmu))
ddl_lon_dt=np.zeros((Nt_plot,Nmu))
ddA_dt=np.zeros((Nt_plot,Nmu))

rsurf=np.zeros((Nt_plot,Nmu))

dLdt=gradient2(time,CL_plot)

count=-1
for i in np.arange(Nt_plot):
    count+=1
    #print(i, np.size(steps),count)
    if (i*1.0/Nt_plot*100)>checkpoints[0]:
        print(i*1.0/Nt_plot*100, '%')
        checkpoints=checkpoints[1:]
    
    temp_data=Hdatabase.interp_database(CL_plot[i],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1],\
                                       [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]],flag_extrap=1)

    temp=np.asarray(temp_data[16])

    dl_lat[count,:]=temp[:,0]
    dl_lon[count,:]=temp[:,1]
    dA[count,:]=temp[:,2]
    
    ddl_lat_dL[count,:]=temp[:,3]
    ddl_lon_dL[count,:]=temp[:,4]
    ddA_dL[count,:]=temp[:,5]
    
    ddl_lat_dt[count,:]=temp[:,3]*dLdt[count]
    ddl_lon_dt[count,:]=temp[:,4]*dLdt[count]
    ddA_dt[count,:]=temp[:,5]*dLdt[count]
    
    rsurf[i,:]=np.asarray(temp_data[17])
    
print('end')

begin
1.2 %
2.1999999999999997 %
3.2 %
4.2 %
5.2 %
6.2 %
7.000000000000001 %
8.200000000000001 %
9.2 %
10.2 %
11.200000000000001 %
12.2 %
13.200000000000001 %
14.000000000000002 %
15.2 %
16.2 %
17.2 %
18.2 %
19.2 %
20.200000000000003 %
21.2 %
22.2 %
23.200000000000003 %
24.2 %
25.2 %
26.200000000000003 %
27.200000000000003 %
28.000000000000004 %
29.2 %
30.2 %
31.2 %
32.2 %
33.2 %
34.2 %
35.199999999999996 %
36.199999999999996 %
37.2 %
38.2 %
39.2 %
40.2 %
41.199999999999996 %
42.199999999999996 %
43.2 %
44.2 %
45.2 %
46.2 %
47.199999999999996 %
48.199999999999996 %
49.2 %
50.2 %
51.2 %
52.2 %
53.2 %
54.2 %
55.00000000000001 %
56.00000000000001 %
57.199999999999996 %
58.199999999999996 %
59.199999999999996 %
60.199999999999996 %
61.199999999999996 %
62.2 %
63.2 %
64.2 %
65.2 %
66.2 %
67.2 %
68.2 %
69.19999999999999 %
70.19999999999999 %
71.2 %
72.2 %
73.2 %
74.2 %
75.2 %
76.2 %
77.2 %
78.2 %
79.2 %
80.2 %
81.2 %
82.19999999999999 %
83.2 %
84.2 %
85.2 %
86.2 %
87.2 %
88.2 %
89.2 %
90.2 %

### Run through and plot all the slides

In [9]:
print('begin')

#################################################
#version with a plot of AM and semi-major axis and latitudinal deformation
#now in a square arangement
#################################################

boxlim=18
z_plot=np.linspace(-boxlim/2, boxlim/2, num=Ncont)
x_plot=np.linspace(-boxlim/2, boxlim/2, num=Ncont)

XXX, ZZZ = np.meshgrid(x_plot, z_plot)

#do the first one twice to make sure the graphics are working
temp=np.append(0,np.arange(np.size(time))[0:])
for k in temp[0:]:
    print(k+1, str(time[k]/1E6)+' Myr', str((k+1)/Nt_plot*100)+'%')
    #loop over all points on the surface
    Nrow_data=Ncont
    x_data=np.zeros(Hdatabase.parr[0].Nmu*Nrow_data*2)
    z_data=np.zeros(Hdatabase.parr[0].Nmu*Nrow_data*2)
    ddl_lon_dt_data=np.zeros(Hdatabase.parr[0].Nmu*Nrow_data*2)
    

    for i in np.arange(Hdatabase.parr[0].Nmu):
        x=rsurf[k,i]*np.sin(np.arccos(Hdatabase.parr[0].mu[i]))
        z=rsurf[k,i]*Hdatabase.parr[0].mu[i]
        x_data[2*i*Nrow_data:(i+1)*2*Nrow_data]=np.append(np.linspace(-x, x, Nrow_data),np.linspace(-x, x, Nrow_data))
        z_data[2*i*Nrow_data:(i+1)*2*Nrow_data]=np.append(np.ones(Nrow_data)*z,-np.ones(Nrow_data)*z)
        ddl_lon_dt_data[2*i*Nrow_data:(i+1)*2*Nrow_data]=np.ones(2*Nrow_data)*ddl_lon_dt[k,i]
        
    DLON=griddata(((x_data.flatten()/1E6,z_data.flatten()/1E6)),ddl_lon_dt_data.flatten()*np.pi/180.0,(XXX,ZZZ),fill_value=np.nan)#, method='linear')
    
    #initialise the figure
    fig = plt.figure(figsize=(7.7,5.5))
    gs0 = gridspec.GridSpec(1, 2, width_ratios=[0.9,1])
    
    gs00 = gridspec.GridSpecFromSubplotSpec(3, 2,
                                            width_ratios=[1,0.05],
                                            height_ratios=[1,1,1.65],
                                    subplot_spec=gs0[0])
    gs01 = gridspec.GridSpecFromSubplotSpec(3, 2,
                                            width_ratios=[1,0.05],
                                            height_ratios=[0.5,2,0.5],
                                    subplot_spec=gs0[1])

    
    ax=[[]]
    ax[0].append(plt.subplot(gs00[0]))
    ax[0].append(plt.subplot(gs00[2], sharex=ax[0][0]))
    ax[0].append(plt.subplot(gs01[2]))
    ax[0].append(plt.subplot(gs00[4], sharex=ax[0][0]))
#         ax[0].append(plt.subplot(gs02[0]))

    ax_col=[[]]
    ax_col[0].append(plt.subplot(gs01[3]))
    
    font = {
    'family' : 'Helvetica',
            'weight' : 'normal',
            'size'   : 8}
    mpl.rc('font', **font)
    
    col=cmaps.parula([0.15,0.85])
    
    ind=np.where(Ctime<=time[k])[0]
    print(ind[-1])
    ax[0][0].plot(Ctime[ind]/1E6, Ca[ind]/REarth, 'k-', linewidth=1.5)
    ax[0][1].plot(Ctime[ind]/1E6, CL[ind]/LEM, 'k-', linewidth=1.5)
    
    
    ax[0][0].set_xlim([-0.05*time[-1]/1E6,1.05*time[-1]/1E6])
    ax[0][1].set_xlim([-0.05*time[-1]/1E6,1.05*time[-1]/1E6])
    ax[0][3].set_xlim([-0.05*time[-1]/1E6,1.05*time[-1]/1E6])
    
    if k==Nt_plot-1:
        ind_max=-1
        ax[0][3].plot([0,Ctime[ind_max]/1E6],[60E-3,60E-3],':', color=col[1],linewidth=1.0, label='Average subduction') #avererage subduction
        ax[0][3].plot([0,Ctime[ind_max]/1E6],[50E-3,50E-3],':', color=col[0],linewidth=1.0, label='Average ridge') #average mid-ocean ridge
        ax[0][3].plot([0,Ctime[ind_max]/1E6],[15E-3,15E-3],'--', color=col[1],linewidth=1.0, label='Slow subduction') #slow subduction
        ax[0][3].plot([0,Ctime[ind_max]/1E6],[8E-3,8E-3],'--', color=col[0],linewidth=1.0, label='Slow ridge') #slow mid ocean ridge
    else:
        ax[0][3].text(0.04, 0.04, 'C: Longitudinal', horizontalalignment='left',verticalalignment='bottom', fontsize=10,transform=ax[0][3].transAxes, color='k')
    
    
    ax[0][3].plot(Ctime[ind]/1E6, np.absolute(ddl_lon_dt_max[ind]), '-', color=col[0], linewidth=1.5)
    ax[0][3].plot(Ctime[ind]/1E6, np.absolute(ddl_lon_dt_min[ind]), '-', color=col[1], linewidth=1.5)
    
    ax[0][3].set_yscale('log')
    
    
    

    if flag_data==0:


        ax[0][0].set_ylim([3,18])
        ax[0][1].set_ylim([0.45,2.2])
        lat_contours=np.linspace(-6,-0.5,22+1)
        lon_contours=np.linspace(-6,-0.5,22+1)
        Acontours=np.linspace(-0.5,5,22+1)

        lat_labels=np.asarray([-6, -5,-4,-3,-2,-1])
        lon_labels=np.asarray([-6,-5,-4,-3,-2,-1])
        Alabels=np.asarray([0,1,2,3,4,5])
        
        ax[0][3].set_ylim([5E-3,1.2E3])

    elif flag_data==1:

        ax[0][0].set_ylim([3,11])
        ax[0][1].set_ylim([0.5,2.2])
        lat_contours=np.linspace(-6,-0.5,22+1)
        lon_contours=np.linspace(-6,-0.5,22+1)
        Acontours=np.linspace(-0.5,5,22+1)

        lat_labels=np.asarray([-6, -5,-4,-3,-2,-1])
        lon_labels=np.asarray([-6,-5,-4,-3,-2,-1])
        Alabels=np.asarray([0,1,2,3,4,5])
        
        ax[0][3].set_ylim([5E-3,3E2])

    elif flag_data==2:

        ax[0][0].set_ylim([3,12])
        ax[0][1].set_ylim([0.5,2.2])
        lat_contours=np.linspace(-7.5,-2.5,20+1)
        lon_contours=np.linspace(-7.5,-2.5,20+1)
        Acontours=np.linspace(-2,3,20+1)

        lat_labels=np.asarray([-7,-6, -5,-4,-3])
        lon_labels=np.asarray([-7,-6,-5,-4,-3])
        Alabels=np.asarray([-2,-1,0,1,2,3])
        
        ax[0][3].set_ylim([8E-4,4E1])

    
    contour2 = ax[0][2].contourf(XXX, ZZZ, np.log10(np.absolute(DLON)), levels=lat_contours, colors=cmaps.parula_r(np.linspace(1, 0, np.size(lat_contours)-1)))
    contour20 = ax[0][2].contour(XXX, ZZZ, DLON, linewidths=0.5, levels=[0.0], colors=['k'], linestyles='solid')
       
        
    for c in contour2.collections:
        c.set_rasterized(True)
        
    #plot the intial outline
    ax[0][2].plot(rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)
    ax[0][2].plot(-rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)
    ax[0][2].plot(rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,-rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)
    ax[0][2].plot(-rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,-rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)

        
    cbar2=plt.colorbar(contour2, cax=ax_col[0][0], orientation='vertical',ticks=lon_labels)
    
    cbar2.set_label(r'Rate [$\log_{10} (|$m deg$^{-1}$ yr$^{-1}$$|)$]', labelpad=-40)
    
    ax[0][2].set_aspect('equal', adjustable='box')
    
    
    #label axis
    ax[0][0].text(0.04, 0.93, 'A: '+"{:.2f}".format(round(time[k]/1E6,2))+' Myrs', horizontalalignment='left',verticalalignment='top', fontsize=10,transform=ax[0][0].transAxes, color='k')
    ax[0][1].text(0.04, 0.07, 'B', horizontalalignment='left',verticalalignment='bottom', fontsize=10,transform=ax[0][1].transAxes, color='k')
    ax[0][2].text(0.04, 0.96, 'D: Longitudinal', horizontalalignment='left',verticalalignment='top', fontsize=10,transform=ax[0][2].transAxes, color='k')
    

    ax[0][2].set_xlabel('[10$^6$ m]')
    
    ax[0][2].set_ylabel('[10$^6$ m]')
    
    ax[0][0].set_ylabel(r"$a_{\rm Moon}$ [$R_{\rm Earth}$]")
    ax[0][1].set_ylabel(r"AM Earth [$L_{\rm EM}$]")
    ax[0][3].set_ylabel(r"Int. deform. rate [m yr$^{-1}$]")
    ax[0][3].set_xlabel(r"Time [Myrs]")
    
    plt.setp( ax[0][0].get_xticklabels(), visible=False)
    plt.setp( ax[0][1].get_xticklabels(), visible=False)

    for i in np.arange(4):
        ax[0][i].tick_params(direction="in", top=True, right=True)


    ax_col[0][0].tick_params(direction="in")
        
    
    if k==Nt_plot-1:
        if (flag_data==4)|(flag_data==7):
            ax[0][3].legend(frameon=False, handlelength=1.9, loc='upper right',fontsize=7.8)
        else:
            ax[0][3].legend(frameon=False, handlelength=1.9, loc='upper right')

    fig.tight_layout()
    if flag_data==0:
        plt.savefig(output_dir+'/MovieS4_slide_'+str(k+1).zfill(5)+'.png', dpi=600, format='png')
    elif flag_data==1:
        plt.savefig(output_dir+'/MovieS5_slide_'+str(k+1).zfill(5)+'.png', dpi=600, format='png')
    elif flag_data==2:
        plt.savefig(output_dir+'/MovieS6_slide_'+str(k+1).zfill(5)+'.png', dpi=600, format='png')
    
    plt.close(fig)

print('done')


begin
1 0.0 Myr 0.2%
0


/var/folders/mm/ct8d2r3j48bcp_kyghqh_n7r0000gq/T/ipykernel_58259/56137987.py:142: MatplotlibDeprecationWarning: The collections attribute was deprecated in Matplotlib 3.8 and will be removed two minor releases later.
  for c in contour2.collections:


1 0.0 Myr 0.2%
0
2 0.076051333 Myr 0.4%
1599
3 0.152102666 Myr 0.6%
3199
4 0.22815399900000002 Myr 0.8%
4799
5 0.304205332 Myr 1.0%
6399
6 0.380256665 Myr 1.2%
7999
7 0.45630799800000005 Myr 1.4000000000000001%
9599
8 0.532359331 Myr 1.6%
11199
9 0.608410664 Myr 1.7999999999999998%
12799
done
